# **BITCOIN PRICE PREDICTION - XGBOOST MODEL**

Professional-grade implementation with hyperparameter tuning

In [33]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    classification_report, 
    roc_auc_score, 
    roc_curve,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import json
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print(" "*25 + "BITCOIN PRICE PREDICTION")
print(" "*28 + "XGBOOST MODEL")
print("="*80)

                         BITCOIN PRICE PREDICTION
                            XGBOOST MODEL


## CONFIGURATION

In [34]:
CONFIG = {
    'data_path': '../data/features/btc_features_complete.csv',
    'prediction_horizon': '24h',  # Options: '1h', '6h', '24h'
    'percentile_threshold': 65,   # Top/Bottom X% for UP/DOWN classification
    'n_features': 79,             # Number of features to select
    'train_ratio': 0.70,
    'val_ratio': 0.15,
    'test_ratio': 0.15,
    'random_state': 42
}

# XGBoost Hyperparameters (tuned for financial data)
XGBOOST_PARAMS = {
    'n_estimators': 500,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'gamma': 0.1,
    'reg_alpha': 0.1,      # L1 regularization
    'reg_lambda': 1.0,     # L2 regularization
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'random_state': CONFIG['random_state'],
    'n_jobs': -1,
    'tree_method': 'hist'
}

## UTILITY FUNCTIONS

In [35]:
def create_percentile_target(returns, threshold_percentile=65):
    """
    Create balanced target based on percentile thresholds.
    Top X% = UP (1), Bottom X% = DOWN (0), Middle = NEUTRAL (-1, excluded)
    """
    up_threshold = np.percentile(returns.dropna(), threshold_percentile)
    down_threshold = np.percentile(returns.dropna(), 100 - threshold_percentile)
    
    target = pd.Series(index=returns.index, dtype=int)
    target[returns > up_threshold] = 1
    target[returns < down_threshold] = 0
    target[(returns >= down_threshold) & (returns <= up_threshold)] = -1
    
    stats = {
        'up_threshold': up_threshold,
        'down_threshold': down_threshold,
        'up_count': (target == 1).sum(),
        'down_count': (target == 0).sum(),
        'neutral_count': (target == -1).sum()
    }
    
    return target, stats

def select_top_features(X, y, n_features=79):
    """Select top N features using mutual information"""
    X_clean = X.fillna(X.median())
    
    selector = SelectKBest(mutual_info_classif, k=min(n_features, X.shape[1]))
    selector.fit(X_clean, y)
    
    feature_scores = pd.DataFrame({
        'feature': X.columns,
        'score': selector.scores_
    }).sort_values('score', ascending=False)
    
    selected = feature_scores.head(n_features)['feature'].tolist()
    return selected, feature_scores

def calculate_class_weights(y):
    """Calculate balanced class weights"""
    class_counts = np.bincount(y.astype(int))
    total = len(y)
    weights = {
        0: total / (2 * class_counts[0]),
        1: total / (2 * class_counts[1])
    }
    return weights

def print_metrics(y_true, y_pred, y_proba, set_name="Validation"):
    """Print comprehensive evaluation metrics"""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    roc = roc_auc_score(y_true, y_proba)
    
    print(f"\n{'='*60}")
    print(f"  {set_name} Set Metrics")
    print(f"{'='*60}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f} (when predicting UP, how often correct)")
    print(f"  Recall:    {rec:.4f} (% of actual UPs correctly identified)")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"  ROC-AUC:   {roc:.4f} ⭐")
    
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n  Confusion Matrix:")
    print(f"                    Predicted DOWN | Predicted UP")
    print(f"  Actual DOWN:      {cm[0][0]:6d}        | {cm[0][1]:6d}")
    print(f"  Actual UP:        {cm[1][0]:6d}        | {cm[1][1]:6d}")
    
    # Class-specific metrics
    down_precision = cm[0][0] / (cm[0][0] + cm[1][0]) if (cm[0][0] + cm[1][0]) > 0 else 0
    down_recall = cm[0][0] / (cm[0][0] + cm[0][1]) if (cm[0][0] + cm[0][1]) > 0 else 0
    up_precision = cm[1][1] / (cm[0][1] + cm[1][1]) if (cm[0][1] + cm[1][1]) > 0 else 0
    up_recall = cm[1][1] / (cm[1][0] + cm[1][1]) if (cm[1][0] + cm[1][1]) > 0 else 0
    
    print(f"\n  Per-Class Performance:")
    print(f"  DOWN → Precision: {down_precision:.4f} | Recall: {down_recall:.4f}")
    print(f"  UP   → Precision: {up_precision:.4f} | Recall: {up_recall:.4f}")
    print(f"{'='*60}")
    
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 
            'f1': f1, 'roc_auc': roc}

def realistic_backtest(y_true, y_pred, prices, initial_capital=10000, fee=0.001):
    """
    Simulate trading strategy:
    - Buy when model predicts UP
    - Sell/stay in cash when model predicts DOWN
    """
    cash = initial_capital
    btc = 0
    portfolio_values = []
    trades = 0
    
    for i in range(len(y_pred)):
        price = prices.iloc[i]
        
        if y_pred[i] == 1 and btc == 0:  # Buy signal
            btc = (cash * (1 - fee)) / price
            cash = 0
            trades += 1
        elif y_pred[i] == 0 and btc > 0:  # Sell signal
            cash = (btc * price) * (1 - fee)
            btc = 0
            trades += 1
        
        portfolio_value = cash + (btc * price)
        portfolio_values.append(portfolio_value)
    
    # Final portfolio value
    final_value = portfolio_values[-1]
    total_return = ((final_value - initial_capital) / initial_capital) * 100
    
    # Buy & Hold benchmark
    buy_hold_return = ((prices.iloc[-1] - prices.iloc[0]) / prices.iloc[0]) * 100
    
    # Sharpe-like metric (simplified)
    returns = pd.Series(portfolio_values).pct_change().dropna()
    sharpe = (returns.mean() / returns.std() * np.sqrt(365 * 24)) if returns.std() > 0 else 0
    
    return {
        'portfolio_values': portfolio_values,
        'final_value': final_value,
        'total_return': total_return,
        'buy_hold_return': buy_hold_return,
        'outperformance': total_return - buy_hold_return,
        'num_trades': trades,
        'sharpe_ratio': sharpe
    }


## STEP 1: LOAD DATA

In [36]:
print(f"\n[1/11] Loading dataset...")
print(f"  Path: {CONFIG['data_path']}")

df = pd.read_csv(CONFIG['data_path'], index_col=0, parse_dates=True)

print(f"✓ Dataset loaded successfully")
print(f"  Total rows: {df.shape[0]:,}")
print(f"  Total columns: {df.shape[1]}")
print(f"  Date range: {df.index.min()} → {df.index.max()}")
print(f"  Duration: {(df.index.max() - df.index.min()).days} days")


[1/11] Loading dataset...
  Path: ../data/features/btc_features_complete.csv
✓ Dataset loaded successfully
  Total rows: 51,443
  Total columns: 86
  Date range: 2020-01-31 00:00:00 → 2025-12-14 18:00:00
  Duration: 2144 days


## STEP 2: CREATE TARGET VARIABLE

In [37]:
print(f"\n[2/11] Creating target variable ({CONFIG['prediction_horizon']} horizon)...")

target_col = f"future_return_{CONFIG['prediction_horizon']}"
if target_col not in df.columns:
    print(f"❌ Error: Column '{target_col}' not found!")
    print(f"Available columns: {df.columns.tolist()}")
    exit(1)

y_target, target_stats = create_percentile_target(
    df[target_col], 
    threshold_percentile=CONFIG['percentile_threshold']
)

print(f"✓ Target created using {CONFIG['percentile_threshold']}th percentile")
print(f"  UP threshold:   {target_stats['up_threshold']*100:+.3f}%")
print(f"  DOWN threshold: {target_stats['down_threshold']*100:+.3f}%")
print(f"\n  Distribution:")
print(f"  UP (1):      {target_stats['up_count']:6,} samples ({target_stats['up_count']/len(y_target)*100:.1f}%)")
print(f"  DOWN (0):    {target_stats['down_count']:6,} samples ({target_stats['down_count']/len(y_target)*100:.1f}%)")
print(f"  NEUTRAL:     {target_stats['neutral_count']:6,} samples (excluded)")

# Filter out neutral samples
valid_mask = y_target != -1
df_filtered = df[valid_mask].copy()
y_filtered = y_target[valid_mask].copy()

print(f"\n  Final dataset: {len(y_filtered):,} samples")


[2/11] Creating target variable (24h horizon)...
✓ Target created using 65th percentile
  UP threshold:   +0.823%
  DOWN threshold: -0.600%

  Distribution:
  UP (1):      18,005 samples (35.0%)
  DOWN (0):    18,005 samples (35.0%)
  NEUTRAL:     15,433 samples (excluded)

  Final dataset: 36,010 samples


## STEP 3: PREPARE FEATURES

In [38]:
print(f"\n[3/11] Preparing features...")

# Drop target columns and non-predictive features
drop_cols = [
    'future_return_1h', 'future_return_6h', 'future_return_24h',
    'target_direction_1h', 'target_multiclass_1h', 'target_return_1h',
    'Close', 'Open', 'High', 'Low',  # Don't use raw OHLC
    'fear_greed_classification'  # Already have numeric version
]

X = df_filtered.drop(columns=drop_cols, errors='ignore')
X = X.select_dtypes(include=[np.number])

# Handle infinite values
X.replace([np.inf, -np.inf], np.nan, inplace=True)

print(f"✓ Feature preparation complete")
print(f"  Initial features: {X.shape[1]}")
print(f"  Missing values: {X.isnull().sum().sum():,}")


[3/11] Preparing features...
✓ Feature preparation complete
  Initial features: 75
  Missing values: 0


## STEP 4: TRAIN-VAL-TEST SPLIT

In [39]:
print(f"\n[4/11] Chronological data split...")

n = len(X)
train_end = int(n * CONFIG['train_ratio'])
val_end = int(n * (CONFIG['train_ratio'] + CONFIG['val_ratio']))

X_train = X.iloc[:train_end].copy()
X_val = X.iloc[train_end:val_end].copy()
X_test = X.iloc[val_end:].copy()

y_train = y_filtered.iloc[:train_end].copy()
y_val = y_filtered.iloc[train_end:val_end].copy()
y_test = y_filtered.iloc[val_end:].copy()

print(f"✓ Data split complete")
print(f"  Training:   {len(X_train):6,} samples ({CONFIG['train_ratio']*100:.0f}%)")
print(f"  Validation: {len(X_val):6,} samples ({CONFIG['val_ratio']*100:.0f}%)")
print(f"  Test:       {len(X_test):6,} samples ({CONFIG['test_ratio']*100:.0f}%)")

print(f"\n  Training set balance:")
print(f"    UP:   {(y_train == 1).sum():,} ({(y_train == 1).sum()/len(y_train)*100:.1f}%)")
print(f"    DOWN: {(y_train == 0).sum():,} ({(y_train == 0).sum()/len(y_train)*100:.1f}%)")



[4/11] Chronological data split...
✓ Data split complete
  Training:   25,207 samples (70%)
  Validation:  5,401 samples (15%)
  Test:        5,402 samples (15%)

  Training set balance:
    UP:   12,669 (50.3%)
    DOWN: 12,538 (49.7%)


## STEP 5: FEATURE SELECTION

In [40]:
print(f"\n[5/11] Feature selection using mutual information...")

selected_features, feature_scores = select_top_features(
    X_train, y_train, n_features=CONFIG['n_features']
)

print(f"✓ Selected top {len(selected_features)} features:")
print(f"\n  Top 15 features by importance:")
for i, feat in enumerate(selected_features[:15], 1):
    score = feature_scores[feature_scores['feature'] == feat]['score'].values[0]
    print(f"  {i:2d}. {feat:35s} (score: {score:.4f})")

if len(selected_features) > 15:
    print(f"  ... and {len(selected_features) - 15} more")

# Apply feature selection
X_train = X_train[selected_features].copy()
X_val = X_val[selected_features].copy()
X_test = X_test[selected_features].copy()


[5/11] Feature selection using mutual information...
✓ Selected top 75 features:

  Top 15 features by importance:
   1. nvt_ratio                           (score: 0.3559)
   2. tx_count_daily                      (score: 0.3159)
   3. market_price_usd                    (score: 0.3149)
   4. hash_rate_change_30d                (score: 0.3133)
   5. avg_block_size_mb                   (score: 0.3106)
   6. tx_count_change_7d                  (score: 0.3102)
   7. tx_fees_btc                         (score: 0.3100)
   8. sp500_change_7d                     (score: 0.2995)
   9. hash_rate_change_7d                 (score: 0.2967)
  10. NASDAQ                              (score: 0.2937)
  11. SP500                               (score: 0.2921)
  12. mempool_size_bytes                  (score: 0.2849)
  13. total_btc_supply                    (score: 0.2764)
  14. hash_rate_th_s                      (score: 0.2699)
  15. GOLD                                (score: 0.2600)
  ... and 60 m

## STEP 6: DATA PREPROCESSING

In [41]:
print(f"\n[6/11] Data preprocessing...")

# Impute missing values with median
train_medians = X_train.median()
X_train.fillna(train_medians, inplace=True)
X_val.fillna(train_medians, inplace=True)
X_test.fillna(train_medians, inplace=True)

# Scale features using RobustScaler (less sensitive to outliers)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"✓ Preprocessing complete")
print(f"  Imputation: Median values")
print(f"  Scaling: RobustScaler (robust to outliers)")


[6/11] Data preprocessing...
✓ Preprocessing complete
  Imputation: Median values
  Scaling: RobustScaler (robust to outliers)


## STEP 7: CALCULATE CLASS WEIGHTS

In [42]:
print(f"\n[7/11] Calculating class weights...")

class_weights = calculate_class_weights(y_train)
sample_weights = np.array([class_weights[int(y)] for y in y_train])

print(f"✓ Class weights calculated")
print(f"  DOWN (0): {class_weights[0]:.4f}")
print(f"  UP (1):   {class_weights[1]:.4f}")


[7/11] Calculating class weights...
✓ Class weights calculated
  DOWN (0): 1.0052
  UP (1):   0.9948


## STEP 8: TRAIN XGBOOST MODEL

In [43]:
print(f"\n[8/11] Training XGBoost model...")
print(f"\n  Hyperparameters:")
for key, value in XGBOOST_PARAMS.items():
    print(f"    {key:20s}: {value}")

# Create DMatrix for XGBoost (more efficient)
dtrain = xgb.DMatrix(X_train_scaled, label=y_train, weight=sample_weights)
dval = xgb.DMatrix(X_val_scaled, label=y_val)

# Train with early stopping
evals = [(dtrain, 'train'), (dval, 'validation')]
evals_result = {}

print(f"\n  Training progress:")
model = xgb.train(
    XGBOOST_PARAMS,
    dtrain,
    num_boost_round=XGBOOST_PARAMS['n_estimators'],
    evals=evals,
    early_stopping_rounds=50,
    evals_result=evals_result,
    verbose_eval=100
)

print(f"\n✓ Training complete")
print(f"  Best iteration: {model.best_iteration}")
print(f"  Best validation AUC: {model.best_score:.4f}")


[8/11] Training XGBoost model...

  Hyperparameters:
    n_estimators        : 500
    max_depth           : 6
    learning_rate       : 0.05
    subsample           : 0.8
    colsample_bytree    : 0.8
    min_child_weight    : 5
    gamma               : 0.1
    reg_alpha           : 0.1
    reg_lambda          : 1.0
    objective           : binary:logistic
    eval_metric         : auc
    random_state        : 42
    n_jobs              : -1
    tree_method         : hist

  Training progress:
[0]	train-auc:0.71703	validation-auc:0.51789
[100]	train-auc:0.97118	validation-auc:0.52990
[131]	train-auc:0.97949	validation-auc:0.53423

✓ Training complete
  Best iteration: 81
  Best validation AUC: 0.5366


## STEP 9: VALIDATION EVALUATION

In [44]:
print(f"\n[9/11] Evaluating on validation set...")

y_val_pred_proba = model.predict(dval)
y_val_pred = (y_val_pred_proba > 0.5).astype(int)

val_metrics = print_metrics(y_val, y_val_pred, y_val_pred_proba, "Validation")



[9/11] Evaluating on validation set...

  Validation Set Metrics
  Accuracy:  0.5332
  Precision: 0.6128 (when predicting UP, how often correct)
  Recall:    0.2539 (% of actual UPs correctly identified)
  F1 Score:  0.3590
  ROC-AUC:   0.5342 ⭐

  Confusion Matrix:
                    Predicted DOWN | Predicted UP
  Actual DOWN:        2174        |    446
  Actual UP:          2075        |    706

  Per-Class Performance:
  DOWN → Precision: 0.5116 | Recall: 0.8298
  UP   → Precision: 0.6128 | Recall: 0.2539


## STEP 10: TEST EVALUATION

In [45]:
print(f"\n[10/11] Final evaluation on test set...")

dtest = xgb.DMatrix(X_test_scaled, label=y_test)
y_test_pred_proba = model.predict(dtest)
y_test_pred = (y_test_pred_proba > 0.5).astype(int)

test_metrics = print_metrics(y_test, y_test_pred, y_test_pred_proba, "Test")



[10/11] Final evaluation on test set...

  Test Set Metrics
  Accuracy:  0.5187
  Precision: 0.4832 (when predicting UP, how often correct)
  Recall:    0.2532 (% of actual UPs correctly identified)
  F1 Score:  0.3323
  ROC-AUC:   0.5173 ⭐

  Confusion Matrix:
                    Predicted DOWN | Predicted UP
  Actual DOWN:        2155        |    692
  Actual UP:          1908        |    647

  Per-Class Performance:
  DOWN → Precision: 0.5304 | Recall: 0.7569
  UP   → Precision: 0.4832 | Recall: 0.2532


## BACKTEST

In [46]:
print(f"\n{'='*60}")
print(f"  Backtesting Trading Strategy")
print(f"{'='*60}")

test_prices = df_filtered.loc[y_test.index, 'Close']
backtest_results = realistic_backtest(y_test, y_test_pred, test_prices)

print(f"\n  Initial Capital:        ${backtest_results['final_value']/((backtest_results['total_return']/100)+1):,.2f}")
print(f"  Final Portfolio Value:  ${backtest_results['final_value']:,.2f}")
print(f"  Total Return:           {backtest_results['total_return']:+.2f}%")
print(f"  Buy & Hold Return:      {backtest_results['buy_hold_return']:+.2f}%")
print(f"  Outperformance:         {backtest_results['outperformance']:+.2f}%")
print(f"  Number of Trades:       {backtest_results['num_trades']}")
print(f"  Sharpe Ratio:           {backtest_results['sharpe_ratio']:.2f}")



  Backtesting Trading Strategy

  Initial Capital:        $10,000.00
  Final Portfolio Value:  $4,549.36
  Total Return:           -54.51%
  Buy & Hold Return:      -9.26%
  Outperformance:         -45.25%
  Number of Trades:       494
  Sharpe Ratio:           -3.92


## STEP 11: SAVE MODEL AND ARTIFACTS

In [47]:
print(f"\n[11/11] Saving model and artifacts...")

os.makedirs('../models', exist_ok=True)

# Save XGBoost model
model.save_model('../models/xgboost_btc.json')

# Save preprocessors
joblib.dump(scaler, '../models/xgboost_scaler.pkl')
joblib.dump(train_medians, '../models/xgboost_medians.pkl')
joblib.dump(selected_features, '../models/xgboost_features.pkl')

# Save comprehensive metrics
results = {
    'config': CONFIG,
    'hyperparameters': XGBOOST_PARAMS,
    'target_stats': {k: float(v) if isinstance(v, (int, float, np.number)) else v 
                     for k, v in target_stats.items()},
    'validation_metrics': val_metrics,
    'test_metrics': test_metrics,
    'backtest_results': {
        'total_return': float(backtest_results['total_return']),
        'buy_hold_return': float(backtest_results['buy_hold_return']),
        'outperformance': float(backtest_results['outperformance']),
        'num_trades': int(backtest_results['num_trades']),
        'sharpe_ratio': float(backtest_results['sharpe_ratio'])
    },
    'selected_features': selected_features,
    'feature_importance': feature_scores.head(25).to_dict('records')
}

with open('../models/xgboost_results.json', 'w') as f:
    json.dump(results, f, indent=4)

print(f"✓ All artifacts saved to '../models/' directory")


[11/11] Saving model and artifacts...
✓ All artifacts saved to '../models/' directory


## FEATURE IMPORTANCE ANALYSIS

In [48]:
print(f"\n{'='*60}")
print(f"  Feature Importance Analysis")
print(f"{'='*60}")

importance_dict = model.get_score(importance_type='gain')
importance_df = pd.DataFrame([
    {'feature': selected_features[int(k.replace('f', ''))], 'importance': v}
    for k, v in importance_dict.items()
]).sort_values('importance', ascending=False)

print(f"\n  Top 10 Most Important Features (by gain):")
for i, row in importance_df.head(10).iterrows():
    print(f"  {row['feature']:35s} {row['importance']:8.0f}")


  Feature Importance Analysis

  Top 10 Most Important Features (by gain):
  difficulty                                47
  total_btc_supply                          44
  EMA_26                                    40
  GOLD                                      36
  DXY                                       35
  BBL_20_2.0_2.0                            34
  BBU_20_2.0_2.0                            34
  market_price_usd                          33
  BBM_20_2.0_2.0                            33
  VIX                                       33


# FINAL SUMMARY

In [49]:
print(f"\n{'='*80}")
print(f" "*30 + "TRAINING COMPLETE")
print(f"{'='*80}")

print(f"\n📊 PERFORMANCE SUMMARY:")
print(f"\n  Validation Set:")
print(f"    ROC-AUC:   {val_metrics['roc_auc']:.4f}")
print(f"    Accuracy:  {val_metrics['accuracy']:.4f}")
print(f"    F1 Score:  {val_metrics['f1']:.4f}")

print(f"\n  Test Set:")
print(f"    ROC-AUC:   {test_metrics['roc_auc']:.4f} ⭐")
print(f"    Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"    F1 Score:  {test_metrics['f1']:.4f}")

print(f"\n  Backtesting:")
print(f"    Strategy Return:  {backtest_results['total_return']:+.2f}%")
print(f"    Outperformance:   {backtest_results['outperformance']:+.2f}%")

print(f"\n💡 INTERPRETATION:")
if test_metrics['roc_auc'] >= 0.65:
    print(f"  ✅ EXCELLENT - Strong predictive power!")
    print(f"     Model shows significant edge. Consider live testing.")
elif test_metrics['roc_auc'] >= 0.60:
    print(f"  ✓ GOOD - Solid predictive ability")
    print(f"    Model has potential. Refine with ensemble methods.")
elif test_metrics['roc_auc'] >= 0.55:
    print(f"  ⚠️  MARGINAL - Weak but detectable signal")
    print(f"     May not be profitable after transaction costs.")
else:
    print(f"  ❌ POOR - No meaningful predictive power")
    print(f"     Try different features, timeframes, or approaches.")

print(f"\n🎯 NEXT STEPS:")
print(f"  1. Analyze feature importance to understand what drives predictions")
print(f"  2. Experiment with different prediction horizons (1h, 6h, 24h)")
print(f"  3. Try ensemble: Combine XGBoost with LSTM or Random Forest")
print(f"  4. Implement more sophisticated trading strategies")
print(f"  5. Add risk management (position sizing, stop-loss)")

print(f"\n{'='*80}\n")


                              TRAINING COMPLETE

📊 PERFORMANCE SUMMARY:

  Validation Set:
    ROC-AUC:   0.5342
    Accuracy:  0.5332
    F1 Score:  0.3590

  Test Set:
    ROC-AUC:   0.5173 ⭐
    Accuracy:  0.5187
    F1 Score:  0.3323

  Backtesting:
    Strategy Return:  -54.51%
    Outperformance:   -45.25%

💡 INTERPRETATION:
  ❌ POOR - No meaningful predictive power
     Try different features, timeframes, or approaches.

🎯 NEXT STEPS:
  1. Analyze feature importance to understand what drives predictions
  2. Experiment with different prediction horizons (1h, 6h, 24h)
  3. Try ensemble: Combine XGBoost with LSTM or Random Forest
  4. Implement more sophisticated trading strategies
  5. Add risk management (position sizing, stop-loss)


